# U-Net H5 Dataset Visualizer
Interactive visualization of paired raw / binary-mask patches from the unified SmartPatches H5.

Reads `/train/raw` + `/train/mask`. When `mask` is absent (the generator was run without `binary_mask_dir`), falls back to deriving binary from `label > 0` on the fly — same contract `H5UNetDataset` enforces at training time.

Patches are stored already whole-volume percentile-normalised (CARE-style), so `raw` should land in roughly `[0, 1]`. If you see big-integer ranges, the H5 was generated with the old code path — regenerate before training.

In [1]:
import numpy as np
import h5py
import matplotlib.pyplot as plt
from ipywidgets import IntSlider, Dropdown, VBox, Output, Label
import ipywidgets as widgets
from IPython.display import display, clear_output

In [2]:
# Set the path to your SmartPatches H5 file.
H5_PATH = "/mnt/jean-zay/segmentation_training/xenopus_segmentation.h5"

In [3]:
# Open H5 file and inspect structure
h5f = h5py.File(H5_PATH, 'r')

print("H5 File Structure:")
print(f"Root keys: {list(h5f.keys())}")
for split in h5f.keys():
    print(f"\n{split}/:")
    for key in h5f[split].keys():
        ds = h5f[split][key]
        sample = ds[0]
        print(f"  {key}: {ds.shape}, dtype={ds.dtype}, "
              f"min={sample.min():.4f}, max={sample.max():.4f}")

H5 File Structure:
Root keys: ['train', 'val']

train/:
  label: (35, 16, 256, 256), dtype=int32, min=0.0000, max=308.0000
  mask: (35, 16, 256, 256), dtype=uint8, min=0.0000, max=1.0000
  raw: (35, 16, 256, 256), dtype=float32, min=0.0000, max=1.0000

val/:
  label: (46, 16, 256, 256), dtype=int32, min=0.0000, max=179.0000
  mask: (46, 16, 256, 256), dtype=uint8, min=0.0000, max=1.0000
  raw: (46, 16, 256, 256), dtype=float32, min=0.0000, max=1.0000


In [4]:
class UNetH5Visualizer:
    """3D-aware raw / mask viewer for the unified SmartPatches H5.

    Handles both 2D and 3D patches transparently — the Z slider is
    hidden for 2D data. When ``mask`` is absent from the split,
    binary is derived from ``label > 0`` on the fly so the viewer
    shows exactly what ``H5UNetDataset`` will see at training time.
    """

    def __init__(self, h5_file):
        self.h5f = h5_file
        self.current_split = 'train'
        self._bind_split(self.current_split)

        self.output = Output()
        self.stats_output = Output()

    def _bind_split(self, split):
        grp = self.h5f[split]
        self.raw = grp['raw']
        # Prefer pre-computed binary mask; fall back to derive from label.
        self.mask = grp['mask'] if 'mask' in grp else None
        self.label = grp['label'] if 'label' in grp else None
        self.n_samples = self.raw.shape[0]
        self.is_3d = self.raw.ndim == 4  # (N, Z, Y, X) for 3D, (N, Y, X) for 2D
        self.n_z = self.raw.shape[1] if self.is_3d else 1

    def switch_split(self, split):
        self.current_split = split
        self._bind_split(split)

    def _get_mask_patch(self, sample_idx):
        if self.mask is not None:
            return (self.mask[sample_idx] > 0).astype(np.uint8)
        if self.label is not None:
            return (self.label[sample_idx] > 0).astype(np.uint8)
        return np.zeros_like(self.raw[sample_idx], dtype=np.uint8)

    def visualize(self, sample_idx, z_idx, show_overlay):
        with self.output:
            clear_output(wait=True)

            raw_patch = self.raw[sample_idx]
            mask_patch = self._get_mask_patch(sample_idx)

            if self.is_3d:
                raw_slice = raw_patch[z_idx]
                mask_slice = mask_patch[z_idx]
                z_label = f", Z={z_idx}"
            else:
                raw_slice = raw_patch
                mask_slice = mask_patch
                z_label = ""

            ncols = 3 if show_overlay else 2
            fig, axes = plt.subplots(1, ncols, figsize=(5 * ncols, 5))

            axes[0].imshow(raw_slice, cmap='gray')
            axes[0].set_title(f'Raw — Sample {sample_idx}{z_label}')
            axes[0].axis('off')

            axes[1].imshow(mask_slice, cmap='Reds', vmin=0, vmax=1)
            mask_src = 'mask' if self.mask is not None else 'label>0'
            fg_frac = float(mask_slice.mean()) * 100
            axes[1].set_title(f'Binary ({mask_src}) — fg {fg_frac:.1f}%')
            axes[1].axis('off')

            if show_overlay:
                axes[2].imshow(raw_slice, cmap='gray')
                mask_rgba = np.zeros(mask_slice.shape + (4,), dtype=np.float32)
                mask_rgba[..., 0] = 1.0  # red channel
                mask_rgba[..., 3] = np.where(mask_slice > 0, 0.4, 0.0)
                axes[2].imshow(mask_rgba)
                axes[2].set_title('Overlay (raw + mask)')
                axes[2].axis('off')

            plt.tight_layout()
            plt.show()

        with self.stats_output:
            clear_output(wait=True)
            print("=" * 60)
            print(f"Patch {sample_idx} — {self.current_split} split")
            print("=" * 60)
            print(f"Raw  shape: {raw_patch.shape}, dtype: {raw_patch.dtype}")
            print(f"Raw  — min: {raw_patch.min():.4f}, max: {raw_patch.max():.4f}, "
                  f"mean: {raw_patch.mean():.4f}, std: {raw_patch.std():.4f}")
            print(f"Mask shape: {mask_patch.shape}, dtype: {mask_patch.dtype}")
            print(f"Mask source: {'/'+self.current_split+'/mask' if self.mask is not None else 'derived from label>0'}")
            print(f"Foreground voxels: {int(mask_patch.sum())}/{mask_patch.size} "
                  f"({mask_patch.mean()*100:.2f}%)")
            print("=" * 60)

    def create_widgets(self):
        split_dropdown = Dropdown(
            options=list(self.h5f.keys()),
            value=self.current_split,
            description='Split:'
        )

        sample_slider = IntSlider(
            min=0, max=self.n_samples - 1, step=1, value=0,
            description='Sample:', continuous_update=False
        )

        z_slider = IntSlider(
            min=0, max=max(self.n_z - 1, 0), step=1, value=self.n_z // 2,
            description='Z slice:', continuous_update=False
        )
        if not self.is_3d:
            z_slider.layout.display = 'none'

        overlay_checkbox = widgets.Checkbox(value=True, description='Show Overlay')

        def on_split_change(change):
            self.switch_split(change['new'])
            sample_slider.max = self.n_samples - 1
            sample_slider.value = min(sample_slider.value, self.n_samples - 1)
            z_slider.max = max(self.n_z - 1, 0)
            z_slider.value = min(z_slider.value, max(self.n_z - 1, 0))
            z_slider.layout.display = 'flex' if self.is_3d else 'none'
            self.visualize(sample_slider.value, z_slider.value, overlay_checkbox.value)

        split_dropdown.observe(on_split_change, names='value')

        def update(sample_idx, z_idx, show_overlay):
            self.visualize(sample_idx, z_idx, show_overlay)

        widgets.interactive(
            update,
            sample_idx=sample_slider,
            z_idx=z_slider,
            show_overlay=overlay_checkbox,
        )

        controls = VBox([
            Label(f'Dataset: {self.n_samples} patches, shape: {self.raw.shape}'),
            split_dropdown,
            sample_slider,
            z_slider,
            overlay_checkbox,
        ])

        display(controls)
        display(self.output)
        display(self.stats_output)

        self.visualize(0, self.n_z // 2, True)

In [5]:
viz = UNetH5Visualizer(h5f)
viz.create_widgets()

Output()

Output()